# Eagle5 v2 Corpus Build — Colab GPU

Builds the DeepSeek-V2-Lite-Chat calibration corpus for Eagle5 v2 draft-head training. The output (~94 parquet shards, ~1.5 GB total at int8 quantize) lands on **your Google Drive** so it survives Colab session disconnects.

## How to use

1. Open this notebook in Colab: `File → Upload notebook → eagle5_v2_corpus.ipynb`.
2. Set the runtime to GPU: `Runtime → Change runtime type → T4 GPU` (free) or `V100/A100` (Pro).
3. Run cells 1–7 in order. Watch for the `[overnight] GPU=...` print to confirm you got a GPU.
4. Cell 5 is the JavaScript keepalive (non-blocking — runs in the browser). Cell 6 is the corpus build (~5-6 hr on T4, ~2 hr on V100, ~1 hr on A100). Resumable: shards are written atomically, so a session disconnect just resumes from the last completed shard on the next run.
5. **Single-tab workflow:** Run Cell 5 first (JS keepalive, non-blocking) then Cell 6 (corpus build). The JS persists in the browser regardless of what Python is doing — no second tab needed.
6. After completion (Cell 7 verifies), the corpus is on Drive at `MyDrive/dismantle/v2_lite_corpus/`. Download to laptop and run training locally with MLX.

## Hardware fit

DeepSeek-V2-Lite is 16B params (~32 GB fp16). On T4 (16 GB VRAM) we use 4-bit nf4 quantization via `bitsandbytes` to fit. On V100 (16 GB) same story. On A100 (40 GB) we run native fp16 with bigger batch.

In [ ]:
# Cell 1 — Mount Drive + verify GPU
from google.colab import drive
drive.mount('/content/drive')

import torch, subprocess
assert torch.cuda.is_available(), "No CUDA GPU — set Runtime → Change runtime type → GPU"
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"[overnight] GPU={gpu_name}  VRAM={vram_gb:.1f} GB")
print(subprocess.check_output(['nvidia-smi']).decode())

In [ ]:
# Cell 2 — Install missing deps (bitsandbytes for 4-bit load)
!pip install -q 'bitsandbytes>=0.43' 'accelerate>=1.0'
import bitsandbytes, accelerate, transformers, datasets, pyarrow
print(f"bitsandbytes {bitsandbytes.__version__}  accelerate {accelerate.__version__}")
print(f"transformers {transformers.__version__}  datasets {datasets.__version__}")

In [ ]:
# Cell 3 — Clone dismantle repo
import os
if not os.path.exists('/content/dismantle'):
    !cd /content && git clone --depth 1 https://github.com/joshuahickscorp/dismantle.git
else:
    !cd /content/dismantle && git pull --ff-only
%cd /content/dismantle
!ls tools/training/build_corpus.py

In [ ]:
# Cell 4 — Patch HF cache for transformers v5 compatibility
#
# DeepSeek-V2-Lite-Chat's modeling_deepseek.py imports `is_torch_fx_available`
# from transformers.utils.import_utils, which was removed in transformers v5.
# Colab ships transformers v5.x, so the load fails. We trigger an
# AutoModelForCausalLM load (which downloads the modeling .py before trying
# to import it), catch the expected ImportError, then patch the cached file
# so the next load succeeds.
#
# Must run BEFORE Cell 4 (the corpus build) in every fresh Colab session.
# Idempotent — re-running is safe.

import glob
from transformers import AutoModelForCausalLM
import transformers
print(f"transformers {transformers.__version__}")

try:
    AutoModelForCausalLM.from_pretrained(
        "deepseek-ai/DeepSeek-V2-Lite-Chat",
        trust_remote_code=True,
    )
    print("(model load unexpectedly succeeded — already patched?)")
except ImportError as e:
    print(f"(triggered cache via expected ImportError: {str(e)[:80]}…)")
except Exception as e:
    print(f"(load errored with {type(e).__name__}; checking if file cached anyway)")

candidates = glob.glob(
    "/root/.cache/huggingface/modules/transformers_modules/**/modeling_deepseek.py",
    recursive=True)
assert candidates, "modeling_deepseek.py STILL not in HF cache — investigate with `!find /root/.cache/huggingface -name modeling_deepseek.py`"
target = candidates[0]
print(f"patching: {target}")

src = open(target).read()
needle = "from transformers.utils.import_utils import is_torch_fx_available"
if needle not in src:
    print("⚠️  needle not found — already patched or upstream changed (likely safe)")
else:
    patched = src.replace(needle, """try:
    from transformers.utils.import_utils import is_torch_fx_available
except ImportError:
    def is_torch_fx_available():
        return False""")
    open(target, "w").write(patched)
    print("✅ patched — Cell 5 will now load past the import error")

In [ ]:
# Cell 5 — START KEEPALIVE (non-blocking, single-tab)
#
# Injects a JavaScript snippet that simulates user activity every 60 sec by
# clicking the Colab connect button. JS runs in the BROWSER, not the Python
# kernel, so Cell 6 (corpus build) runs in parallel — no second tab needed.
#
# Run this ONCE before Cell 6. It persists for the lifetime of the browser
# tab (until you close or reload it). Output in browser DevTools console
# (F12 → Console) shows ticks if you want to verify.

from IPython.display import display, Javascript
display(Javascript("""
function _dismantle_keepalive() {
  try {
    const btn = document.querySelector("colab-connect-button");
    if (btn && btn.shadowRoot) {
      const inner = btn.shadowRoot.querySelector("#connect");
      if (inner) inner.click();
    }
    document.dispatchEvent(new KeyboardEvent("keydown", {key: "Shift"}));
    console.log("[dismantle keepalive] tick " + new Date().toISOString());
  } catch (e) {
    console.warn("[dismantle keepalive] error", e);
  }
}
_dismantle_keepalive();                       // initial tick
setInterval(_dismantle_keepalive, 60 * 1000); // every 60 sec
"""))
print("✅ keepalive injected — clicks connect-button every 60 sec via JS")
print("   Non-blocking. Proceed to Cell 6 in this same tab.")
print("   (F12 → Console to see ticks if curious; reload page to stop)")

In [ ]:
# Cell 6 — RUN CORPUS BUILD (MAXED — 20000 seqs × 4096 max-tokens)
#
# Output lands on Drive so it survives disconnects. --load-4bit auto-detected
# for T4/V100; native fp16 on A100. Resumable: --skip-existing detects
# already-completed shards on Drive.
#
# Cell 5 keepalive runs in the browser; this cell can run 5-6 hr in the
# same tab without idle disconnect.

# CRITICAL: set PYTORCH_ALLOC_CONF=expandable_segments:True BEFORE python
# starts. Avoids fragmentation OOM (V2-Lite at 4-bit barely fits T4's 14.5
# GB usable; the 20 MiB-shortfall load failure was a fragmentation issue).
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

OUT = "/content/drive/MyDrive/dismantle/v2_lite_corpus"
os.makedirs(OUT, exist_ok=True)
existing = len([f for f in os.listdir(OUT) if f.endswith(".parquet")])
print(f"[overnight] existing shards on Drive: {existing} / 625 target (20k seqs / 32 per shard)")

import torch
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
use_4bit = vram_gb < 30
load_flag = "--load-4bit" if use_4bit else ""
# Batch: keep small on T4 to leave room for activations.
#   T4 ~14.5 GB usable: 4-bit V2-Lite ~12 GB → 2.5 GB for activations
#                       batch=4 × 4096 tokens × ~0.1 GB = 1.6 GB activations
#   A100 40 GB: native fp16 model ~32 GB → 8 GB for activations → batch=16
batch = 4 if vram_gb < 20 else 8 if vram_gb < 32 else 16
# CUDA VRAM cap: force CPU offload of overflow layers. Requires
# llm_int8_enable_fp32_cpu_offload=True in BnbConfig (added in
# build_corpus.py) — without that, bnb 4-bit IGNORES the cap.
# 12 GiB cap: ~2.5 GiB spills to CPU as fp32 (~10 GB CPU RAM used);
# Colab free has 12.7 GB CPU RAM, leaving ~2 GB margin.
vram_cap_flag = "--cuda-max-vram-gb 12" if vram_gb < 20 else ""
print(f"[overnight] batch={batch}  load={'4-bit nf4' if use_4bit else 'fp16'}  "
      f"vram_cap={'12 GiB → CPU spillover' if vram_cap_flag else 'unrestricted'}")

!PYTORCH_ALLOC_CONF=expandable_segments:True python tools/training/build_corpus.py \
  --model deepseek-ai/DeepSeek-V2-Lite-Chat \
  --dataset HuggingFaceH4/ultrachat_200k \
  --max-sequences 20000 \
  --batch-size {batch} \
  --max-tokens-per-seq 4096 \
  --shard-size 32 \
  --device cuda \
  --dtype float16 \
  --capture all \
  {load_flag} \
  {vram_cap_flag} \
  --out {OUT}

In [ ]:
# Cell 7 — Verify + summarize when corpus is done
import os, glob
OUT = '/content/drive/MyDrive/dismantle/v2_lite_corpus'
shards = sorted(glob.glob(f'{OUT}/*.parquet'))
if not shards:
    print(f"⚠️  no shards at {OUT} — corpus build hasn't completed")
else:
    total = sum(os.path.getsize(s) for s in shards) / 1e6
    print(f"✅ {len(shards)} shards on Drive, total {total:.1f} MB")
    print(f"   first: {os.path.basename(shards[0])}")
    print(f"   last:  {os.path.basename(shards[-1])}")
    print()
    print("Download to laptop:")
    print(f"   1. Open Drive: drive.google.com → MyDrive/dismantle/v2_lite_corpus/")
    print(f"   2. Right-click the folder → Download (Drive zips it)")
    print(f"   3. On laptop:  unzip → mv to ~/Downloads/dismantle/artifacts/calibration/v2_lite_corpus/")
    print(f"   4. Run laptop chain:  tools/bench/overnight_eagle5_2026_05_26.sh")
    print(f"      (chain detects existing corpus and skips straight to training)")

In [ ]:
# Cell 8 (OPTIONAL) — Run training on Colab too
#
# If you'd rather not download the corpus + train on laptop, you can train
# right here on Colab. Caveat: eagle5_train.py uses MLX which is Apple-only;
# this cell ports the training to PyTorch on the fly. ~30 min on T4.
#
# Generally not worth it — MLX training on laptop is faster than PyTorch on
# T4 once the corpus is local. Skip this cell unless you need the head
# without downloading the corpus.

print("Skipping — train on laptop with MLX. See README in colab/ folder.")